# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is primarily a **ranking / scoring** problem, implemented under the hood as **binary classification**. The end deliverable isn't a single yes/no label per page — it's an ordered queue: "review this page before that one." I get there by training a classifier that outputs a probability ("how likely is this page to be declining?"), then I sort pages by that probability to build the queue. This matches how the starter pipeline itself works and how it's evaluated — with Precision@K, a ranking metric, not plain accuracy.

It is **not** clustering (I'm not looking for unlabeled groups of similar pages — that's Lane 3's job) and it is not pure classification in the sense of "does this page belong to class A" being the final output — the probability itself is the useful part, because it lets me rank.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For this notebook I'm using the starter's proxy label: `is_declining_label = (trend_direction == "down")`. This comes from a **defined rule on the current window**, not a true future outcome — `trend_direction` is calculated by comparing the last 30 days to the previous 30 days, so it's already known at the time I'd be scoring the page. That makes it a reasonable proxy to sketch the task with now, but it is exactly the weakness the lane guide warns about: it's a bucket calculated from the current window, not a genuinely future-looking label.

For the actual capstone, the stronger version is a real future-observed outcome: **features from the prior 90 days → decline (or recovery) over the next 30 days** — a label that didn't exist yet at prediction time, which is what makes ML actually useful here instead of just re-describing the present.


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df["trend_direction"].value_counts())
print()
print(f"is_declining_label rate: {df['is_declining_label'].mean()*100:.1f}% of {len(df)} pages")


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label rate: 54.2% of 30000 pages


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** A content reviewer can realistically look at roughly 50 pages a week — that's the real decision capacity this output has to match. Precision@50 asks: of the top 50 pages my ranking puts first, what fraction are actually declining? That's a direct measure of whether the reviewer's limited time gets spent well, which plain accuracy or ROC AUC would not capture — a model can have great overall accuracy while still filling the top 50 with false alarms if the ranking at the very top is weak.

From Week 1: the hand-written baseline scored **0.24** on this metric; the trained random forest scored **0.74** — roughly 3x more of the reviewer's first 50 checks would actually be worth their time.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Precision@50 reference numbers, verified in outputs/model_report.md from Week 1:
print("baseline rules   Precision@50: 0.240  (~12 of top 50 correct)")
print("random forest    Precision@50: 0.740  (~37 of top 50 correct)")


baseline rules   Precision@50: 0.240  (~12 of top 50 correct)
random forest    Precision@50: 0.740  (~37 of top 50 correct)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page** (`content_id`), scored at one point in time. Each page belongs to one client (`client_id`) and carries its own search/engagement signals for the current 90-day window, plus the label I'd be predicting.


In [4]:
cols_to_show = [
    "content_id", "client_id", "content_type", "impressions_90d",
    "avg_position", "ctr", "days_since_last_update", "trend_direction",
    "is_declining_label",
]

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df[cols_to_show].head(8)


Shape: 30000 rows, 45 columns


,content_id,client_id,content_type,impressions_90d,avg_position,ctr,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,10.6,0.76,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,20.3,0.05,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,36.5,0.09,20,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,6.2,0.49,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,44.0,0.13,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,8.5,0.03,20,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,7.0,0.00,20,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,21.2,0.06,22,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Two pieces of evidence from this data:

**1. No single signal is a strong enough predictor on its own.** I checked the raw correlation between the declining label and several obvious candidate signals — `days_since_last_update`, `impressions_90d`, `avg_position`, `word_count`, `ctr`. Every one of them is weak in isolation (all under 0.10 in magnitude). A fixed if-statement built on any one of these — or even a few ANDed together — can't capture a pattern that only shows up in the *combination* of several weak signals.

**2. A reasonable-sounding fixed rule misses almost everyone.** The obvious hand-written rule ("stale AND still visible") only catches 16 of the 16,262 declining pages in this sample — about 0.1%. That's not a rule that's slightly imperfect; it's a rule that's blind to the vast majority of the actual pattern. A model that can weigh and combine many weak signals together — the same kind of interaction a decision tree or random forest naturally captures — is what closed that gap to 74% Precision@50 in Week 1.

This is why it's an ML/analysis problem and not just an if-statement: the signal exists, but only when several observed measurements are considered *together*, not one at a time.


In [5]:
for col in ["days_since_last_update", "impressions_90d", "avg_position", "word_count", "ctr"]:
    corr = df[col].corr(df["is_declining_label"])
    print(f"{col}: correlation with declining label = {corr:.3f}")

print()

declining_df = df[df["is_declining_label"] == 1]
stale_visible = declining_df[
    (declining_df["days_since_last_update"] >= 180) &
    (declining_df["impressions_90d"] >= 500)
]
print(f"Fixed rule (stale AND visible) catches {len(stale_visible)} of {len(declining_df)} "
      f"declining pages ({len(stale_visible)/len(declining_df)*100:.1f}%)")


days_since_last_update: correlation with declining label = 0.081
impressions_90d: correlation with declining label = -0.018
avg_position: correlation with declining label = -0.029
word_count: correlation with declining label = 0.090
ctr: correlation with declining label = -0.062

Fixed rule (stale AND visible) catches 16 of 16262 declining pages (0.1%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.